# EbbRAG — Baselines (SPEC-02)

Reference systems for comparison against EbbRAG.

**Baselines:** VanillaRAG · BM25RAG · SelfRAG  
**Requires:** `EbbRAG_01_core.ipynb` already run in this session (or re-run cells 1→7 below)

All baselines implement the same interface as EbbRAG so the experiment loop can call them uniformly.

## 1. Install & imports (skip if already done in core notebook)

In [ ]:
!pip install -q anthropic>=0.25.0 sentence-transformers>=2.7.0 rank_bm25>=0.2.2 datasets>=2.19.0 tqdm

from google.colab import userdata
import os
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

from dataclasses import dataclass, field
from typing import Optional, List
from abc import ABC, abstractmethod
import time, math, pickle
import numpy as np
import anthropic
from sentence_transformers import SentenceTransformer

print('Imports ✓')

## 2. Re-define core classes (Chunk, QueryResult, Embedder, ChunkIndex, Generator)

> Skip this cell if you are continuing from EbbRAG_01_core in the same session.

In [ ]:
@dataclass
class Chunk:
    id: str
    text: str
    embedding: Optional[List[float]] = None
    stability: float = 0.5
    last_retrieved: float = field(default_factory=time.time)
    retrieval_count: int = 0

@dataclass
class QueryResult:
    query_id: str
    query: str
    retrieved_chunks: List[Chunk]
    retrieval_scores: List[float]
    parametric_answer: Optional[str]
    final_answer: str
    latency_ms: float
    tokens_used: int
    em: Optional[int] = None

class Embedder:
    def __init__(self, model_name: str = 'BAAI/bge-small-en-v1.5'):
        print(f'Loading {model_name}...')
        self.model = SentenceTransformer(model_name)
        print('Embedder ✓')
    def embed(self, text: str) -> np.ndarray:
        return self.model.encode(text, normalize_embeddings=True)
    def embed_batch(self, texts: List[str], batch_size: int = 64) -> np.ndarray:
        return self.model.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)

class ChunkIndex:
    def __init__(self, lambda_decay: float = 0.05, epsilon: float = 0.01):
        self.chunks: List[Chunk] = []
        self.embeddings: Optional[np.ndarray] = None
        self.lambda_decay = lambda_decay
        self.epsilon = epsilon
    def add_chunks(self, chunks: List[Chunk], embedder: Embedder):
        embs = embedder.embed_batch([c.text for c in chunks])
        for chunk, emb in zip(chunks, embs):
            chunk.embedding = emb.tolist()
        self.chunks.extend(chunks)
        self.embeddings = np.array([c.embedding for c in self.chunks])
        print(f'Indexed {len(self.chunks)} chunks ✓')
    def search(self, query_emb: np.ndarray, k: int = 5) -> List[tuple]:
        scores = self.embeddings @ query_emb
        top_idx = np.argsort(scores)[::-1][:k]
        return [(self.chunks[i], float(scores[i])) for i in top_idx]
    @classmethod
    def load(cls, path: str) -> 'ChunkIndex':
        with open(path, 'rb') as f: return pickle.load(f)

class Generator:
    def __init__(self, model: str = 'claude-sonnet-4-6', max_tokens: int = 256):
        self.client = anthropic.Anthropic()
        self.model = model
        self.max_tokens = max_tokens
        self._total_tokens = 0
    def generate(self, query: str, chunks: Optional[List[Chunk]] = None) -> tuple:
        if chunks:
            context = '\n\n'.join([f'[{i+1}] {c.text}' for i, c in enumerate(chunks)])
            prompt = f'Answer based on context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:'
        else:
            prompt = f'Answer from memory:\n\nQuestion: {query}\nAnswer:'
        resp = self.client.messages.create(
            model=self.model, max_tokens=self.max_tokens,
            messages=[{'role': 'user', 'content': prompt}]
        )
        tokens = resp.usage.input_tokens + resp.usage.output_tokens
        self._total_tokens += tokens
        return resp.content[0].text.strip(), tokens

embedder = Embedder()
generator = Generator()
print('Core classes ready ✓')

## 3. BaseRAG interface

In [ ]:
class BaseRAG(ABC):
    """All systems implement this interface for uniform experiment loop usage."""

    @abstractmethod
    def query(
        self,
        query: str,
        query_id: str = 'q0',
        ground_truth: Optional[str] = None,
        t_now: Optional[float] = None
    ) -> QueryResult:
        ...

print('BaseRAG interface ✓')

## 4. VanillaRAG

Standard dense retrieval, no temporal weighting. Primary baseline.

> Note: `EbbRAG(use_sr=False, use_ar=False)` should produce identical results — verified in smoke test.

In [ ]:
class VanillaRAG(BaseRAG):
    """
    Standard top-k cosine retrieval RAG.
    No SR scoring, no AR step.
    """
    def __init__(self, index: ChunkIndex, embedder: Embedder,
                 generator: Generator, k: int = 5):
        self.index = index
        self.embedder = embedder
        self.generator = generator
        self.k = k

    def query(self, query: str, query_id: str = 'q0',
              ground_truth: Optional[str] = None,
              t_now: Optional[float] = None) -> QueryResult:
        start = time.time()
        query_emb = self.embedder.embed(query)
        results = self.index.search(query_emb, k=self.k)
        chunks = [c for c, _ in results]
        scores = [s for _, s in results]
        final_answer, tokens = self.generator.generate(query, chunks=chunks)
        em = int(ground_truth.lower().strip() in final_answer.lower()) if ground_truth else None
        return QueryResult(
            query_id=query_id, query=query,
            retrieved_chunks=chunks, retrieval_scores=scores,
            parametric_answer=None, final_answer=final_answer,
            latency_ms=(time.time() - start) * 1000,
            tokens_used=tokens, em=em
        )

print('VanillaRAG ✓')

## 5. BM25RAG

Lexical retrieval baseline. Tests whether SR gains are retrieval-method agnostic.

In [ ]:
from rank_bm25 import BM25Okapi

class BM25RAG(BaseRAG):
    """
    BM25 lexical retrieval + Anthropic generator.
    Does not require embeddings.
    """
    def __init__(self, chunks: List[Chunk], generator: Generator, k: int = 5):
        self.chunks = chunks
        self.generator = generator
        self.k = k
        print('Building BM25 index...')
        tokenized = [c.text.lower().split() for c in chunks]
        self.bm25 = BM25Okapi(tokenized)
        print(f'BM25 index built ({len(chunks)} chunks) ✓')

    def query(self, query: str, query_id: str = 'q0',
              ground_truth: Optional[str] = None,
              t_now: Optional[float] = None) -> QueryResult:
        start = time.time()
        tokens_q = query.lower().split()
        scores = self.bm25.get_scores(tokens_q)
        top_idx = np.argsort(scores)[::-1][:self.k]
        chunks = [self.chunks[i] for i in top_idx]
        top_scores = [float(scores[i]) for i in top_idx]
        final_answer, tokens = self.generator.generate(query, chunks=chunks)
        em = int(ground_truth.lower().strip() in final_answer.lower()) if ground_truth else None
        return QueryResult(
            query_id=query_id, query=query,
            retrieved_chunks=chunks, retrieval_scores=top_scores,
            parametric_answer=None, final_answer=final_answer,
            latency_ms=(time.time() - start) * 1000,
            tokens_used=tokens, em=em
        )

print('BM25RAG ✓')

## 6. SelfRAG (prompt-engineering variant)

Simplified Self-RAG without fine-tuning — uses prompt engineering to simulate:
1. **Retrieve decision:** does this query need external knowledge?
2. **Relevance filter:** is each retrieved chunk relevant?
3. **Support check:** is the answer supported by context?

> ⚠️ Note: This is weaker than the published Self-RAG (Asai et al. 2024) which uses a fine-tuned model. Document this limitation in the paper.

In [ ]:
class SelfRAG(BaseRAG):
    """
    Prompt-engineered Self-RAG (no fine-tuning).
    Uses Claude to decide retrieve/filter/support via explicit prompts.
    """
    def __init__(self, index: ChunkIndex, embedder: Embedder,
                 generator: Generator, k: int = 5):
        self.index = index
        self.embedder = embedder
        self.generator = generator
        self.k = k

    def _should_retrieve(self, query: str) -> bool:
        """Step 1: Does this query need external knowledge?"""
        prompt = (
            f'Does answering this question require looking up external facts, '
            f'or can you answer it from general knowledge?\n\n'
            f'Question: {query}\n\n'
            f'Reply with only YES (need retrieval) or NO (can answer from memory).'
        )
        resp = self.generator.client.messages.create(
            model=self.generator.model, max_tokens=10,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return 'YES' in resp.content[0].text.upper()

    def _filter_relevant(self, query: str, chunks: List[Chunk]) -> List[Chunk]:
        """Step 2: Keep only chunks relevant to the query."""
        relevant = []
        for chunk in chunks:
            prompt = (
                f'Is the following passage relevant to answering this question?\n\n'
                f'Question: {query}\n'
                f'Passage: {chunk.text}\n\n'
                f'Reply with only YES or NO.'
            )
            resp = self.generator.client.messages.create(
                model=self.generator.model, max_tokens=10,
                messages=[{'role': 'user', 'content': prompt}]
            )
            if 'YES' in resp.content[0].text.upper():
                relevant.append(chunk)
        return relevant if relevant else chunks  # fallback: use all if none pass

    def query(self, query: str, query_id: str = 'q0',
              ground_truth: Optional[str] = None,
              t_now: Optional[float] = None) -> QueryResult:
        start = time.time()
        total_tokens = 0

        # Step 1: Decide whether to retrieve
        do_retrieve = self._should_retrieve(query)

        if do_retrieve:
            # Step 2: Retrieve candidates
            query_emb = self.embedder.embed(query)
            results = self.index.search(query_emb, k=self.k)
            candidates = [c for c, _ in results]
            scores = [s for _, s in results]

            # Step 3: Filter for relevance
            chunks = self._filter_relevant(query, candidates)
        else:
            chunks = []
            scores = []

        # Step 4: Generate answer
        final_answer, tok = self.generator.generate(query, chunks=chunks if chunks else None)
        total_tokens += tok

        em = int(ground_truth.lower().strip() in final_answer.lower()) if ground_truth else None

        return QueryResult(
            query_id=query_id, query=query,
            retrieved_chunks=chunks, retrieval_scores=scores[:len(chunks)],
            parametric_answer=None, final_answer=final_answer,
            latency_ms=(time.time() - start) * 1000,
            tokens_used=total_tokens, em=em
        )

print('SelfRAG ✓')

## 7. Smoke test — all baselines

In [ ]:
# Shared toy index
toy_chunks = [
    Chunk(id='c1', text='Hermann Ebbinghaus was a German psychologist who studied memory and forgetting.'),
    Chunk(id='c2', text='The forgetting curve shows how memory retention decreases over time without reinforcement.'),
    Chunk(id='c3', text='Spaced repetition exploits the spacing effect to improve long-term memory retention.'),
]

index = ChunkIndex()
index.add_chunks(toy_chunks, embedder)

systems = {
    'VanillaRAG': VanillaRAG(index=index, embedder=embedder, generator=generator, k=2),
    'BM25RAG':    BM25RAG(chunks=toy_chunks, generator=generator, k=2),
    'SelfRAG':    SelfRAG(index=index, embedder=embedder, generator=generator, k=2),
}

QUERY = 'Who studied the forgetting curve?'
GT    = 'Ebbinghaus'

print(f'Query: {QUERY}\n')
for name, system in systems.items():
    r = system.query(QUERY, query_id=f'smoke-{name}', ground_truth=GT)
    print(f'{name:<12} → answer: {r.final_answer[:60]}')
    print(f'             EM={r.em}  latency={r.latency_ms:.0f}ms  tokens={r.tokens_used}')
    print()

print('Baseline smoke test complete ✓')